# Infra-Bench CLS — CROMA_base Linear Probe (joint Sentinel-1 + Sentinel-2)

Linear-probe evaluation of CROMA's joint SAR + optical encoder on the
Infra-Bench CLS 13-class benchmark. Frozen backbone, trainable linear
head, 3 seeds, spatial split.

## Model

- **CROMA_base** (Fuller, Millard, Green 2023). Repo:
  <https://github.com/antofuller/CROMA>. Weights:
  <https://huggingface.co/antofuller/CROMA>.
- Joint Sentinel-1 + Sentinel-2 encoder pretrained contrastively. Forward
  returns `{'SAR_GAP', 'optical_GAP', 'joint_GAP'}`; this notebook uses
  `joint_GAP` (dim 768) as the LP feature.
- Loader: `wget` fetches `use_croma.py` directly from the repo; weights
  via `huggingface_hub.hf_hub_download`. Only runtime deps beyond
  Colab defaults are `torch` and `einops`.

## Input pipeline

### Band mapping (S2)
Our `.npy` storage: `[B04, B03, B02, B08, B8A, B11, B12, VV, VH]` (indices 0–8).
CROMA expects S2 in order `[B01, B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12, B09]`.

| CROMA slot | Band | Source        |
|-----------:|:-----|:--------------|
| 0          | B01  | **zero-pad**  |
| 1          | B02  | our idx 2     |
| 2          | B03  | our idx 1     |
| 3          | B04  | our idx 0     |
| 4          | B05  | **zero-pad**  |
| 5          | B06  | **zero-pad**  |
| 6          | B07  | **zero-pad**  |
| 7          | B08  | our idx 3     |
| 8          | B8A  | our idx 4     |
| 9          | B11  | our idx 5     |
| 10         | B12  | our idx 6     |
| 11         | B09  | **zero-pad**  |

**5 of 12 CROMA S2 channels are zero-padded** — a documented data-coverage
limitation (we do not fetch B01, B05–B07, or B09 during curation).

### Band mapping (S1)
CROMA expects `[VV, VH]` — pulled directly from our indices `[7, 8]`.

### Normalization
`percentile_normalize` (clip 2nd–98th percentile, rescale to `[0, 1]`)
applied **per modality separately**. A single joint normalization across
the 9 raw channels would let S1's dB range clobber S2 statistics.

### Image size
Native tiles are ~61×61; resized to 120×120 (CROMA's pretraining
resolution).

## Split (shared across all Infra-Bench CLS FMs)

Spatial block-based split loaded from
`data/spatial_split/asset_id_to_split_v1.parquet`. Blocks assign a whole
~0.5° spatial region to the same partition so tiles near a training asset
cannot leak into val or test. Invariant across seeds and across every FM.

## Training protocol

- **25 epochs**, batch 16, AdamW, LR **1e-3**
- Class-weighted CE, weights capped at 10×
- **Frozen backbone** (`requires_grad=False`, `.eval()` mode). Head is
  `InfraBenchClassifier`: `Linear(768 → 13)` with `Dropout(0.1)`.
- **3 seeds** (314, 271, 161). Each seed varies the head init +
  DataLoader shuffle only. CROMA has no augmentation, and features are
  seed-invariant since the backbone is frozen (linear-probe-on-frozen-
  features convention per Chen et al. 2020).
- **Best-val checkpoint restored before the held-out test pass**
  (search this notebook for `BEST_CKPT_BEFORE_TEST`).

## Per-sector F1 definition

Macro-average of per-class F1s for the classes in that sector, computed
on the FULL test set. Return schema: `{n, macro_f1, acc,
per_class_f1_in_sector}` per sector.

## Aggregate output

Combines the 3 seeds into `mean ± std` and `per_seed` arrays for every
metric. Write is gated by `set(SEEDS) == set(FULL_PROTOCOL_SEEDS)`.

## Outputs

- Per-seed: `results/fm_eval_croma_v2_spatial/croma_v2_seed{314,271,161}_results.json`
- Aggregate: `results/fm_eval_croma_v2_spatial/croma_v2_aggregate.json`
- Confusion matrix: `results/fm_eval_croma_v2_spatial/confusion_matrix_croma_v2_aggregate.png`

## Runtime

`SMOKE_ONLY=True` by default. Set to `False` and re-run the training
cells to launch the full 3-seed run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: True
GPU: NVIDIA L4
Memory: 23.7 GB


In [2]:
%%capture
!pip install -q einops huggingface_hub scikit-learn pyproj pyarrow


In [3]:
import os, sys, urllib.request
from pathlib import Path

CROMA_DIR     = Path('/content/croma_repo')
CROMA_DIR.mkdir(exist_ok=True)
USE_CROMA_PY  = CROMA_DIR / 'use_croma.py'
USE_CROMA_URL = 'https://raw.githubusercontent.com/antofuller/CROMA/main/use_croma.py'

if not USE_CROMA_PY.exists():
    print(f'Fetching {USE_CROMA_URL} ...')
    try:
        urllib.request.urlretrieve(USE_CROMA_URL, USE_CROMA_PY)
        print(f'  -> {USE_CROMA_PY}  ({USE_CROMA_PY.stat().st_size:,} bytes)')
    except Exception as e:
        print(f'  direct download failed ({e}); falling back to git clone')
        import subprocess
        subprocess.check_call(['git', 'clone', '--depth', '1', '-q',
                               'https://github.com/antofuller/CROMA.git',
                               str(CROMA_DIR)])
else:
    print(f'Already have {USE_CROMA_PY}')

from huggingface_hub import hf_hub_download
WEIGHT_CANDIDATES = ['CROMA_base.pt', 'croma_base.pt', 'CROMA_base.pth']
CROMA_WEIGHTS = None
last_err = None
for candidate in WEIGHT_CANDIDATES:
    try:
        CROMA_WEIGHTS = hf_hub_download(repo_id='antofuller/CROMA', filename=candidate)
        print(f'Downloaded weights: {candidate} -> {CROMA_WEIGHTS}')
        break
    except Exception as e:
        last_err = e
        continue
if CROMA_WEIGHTS is None:
    raise RuntimeError(f'No CROMA_base weights on HF. Last error: {last_err}')

if str(CROMA_DIR) not in sys.path:
    sys.path.insert(0, str(CROMA_DIR))
from use_croma import PretrainedCROMA
print('use_croma.PretrainedCROMA imported OK')


Already have /content/croma_repo/use_croma.py
Downloaded weights: CROMA_base.pt -> /root/.cache/huggingface/hub/models--antofuller--CROMA/snapshots/0dd28e3d633bd6715856ae9890e8c49360040598/CROMA_base.pt
use_croma.PretrainedCROMA imported OK


In [4]:
# Extract curation code zip to get curation.utils.spatial_blocking.
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Import the spatial split loader. If the zip is older than Phase 1 (no
# spatial_blocking.py yet), fall back to a slim local definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip ...
done.
Could not import (zip is pre-Phase-1): No module named 'curation.utils.spatial_blocking'. Using inline fallback.


In [5]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_croma_v2_spatial'
SPLIT_ARTIFACT_PATH = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':              'water.water_works',   # legacy manifest tag
    'water.water_works':                  'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

# CROMA S2 channel remap (unchanged from v1).
CROMA_S2_FROM_OURS = [-1, 2, 1, 0, -1, -1, -1, 3, 4, 5, 6, -1]
S1_INDICES_IN_OURS = [7, 8]

# Sector partition for v2 per-sector F1.
SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

PERC_LO, PERC_HI = 2.0, 98.0
IMAGE_SIZE = 224
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0
SEEDS      = [161]      # π×100, e×100, φ×100

def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f'Output dir:        {OUTPUT_DIR}')
print(f'Split artifact:    {SPLIT_ARTIFACT_PATH}')
print(f'Training seeds:    {SEEDS}')
print(f'Linear probe:      {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')


Output dir:        /content/drive/MyDrive/infra_fm/results/fm_eval_croma_v2_spatial
Split artifact:    /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
Training seeds:    [161]
Linear probe:      25 epochs, batch 16, lr 0.001


In [6]:
# Same materialize / discovery as v1.
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')

def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


  [DONE]   africa                 energy     already present (949 tiles)
  [DONE]   africa                 telecom    already present (1 tiles)
  [DONE]   africa                 transport  already present (1000 tiles)
  [DONE]   africa                 water      already present (892 tiles)
  [DONE]   asia                   energy     already present (889 tiles)
  [DONE]   asia                   telecom    already present (28 tiles)
  [DONE]   asia                   transport  already present (891 tiles)
  [DONE]   asia                   water      already present (958 tiles)
  [DONE]   australia-oceania      energy     already present (1002 tiles)
  [DONE]   australia-oceania      telecom    already present (12 tiles)
  [DONE]   australia-oceania      transport  already present (1000 tiles)
  [DONE]   australia-oceania      water      already present (1000 tiles)
  [DONE]   central-america        energy     already present (999 tiles)
  [DONE]   central-america        telecom    alread

In [7]:
# Same CROMADataset + wrappers as v1 — split assignment happens later.
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class CROMADataset(Dataset):
    def __init__(self, dataset_root,
                 croma_s2_from_ours=CROMA_S2_FROM_OURS,
                 s1_indices=S1_INDICES_IN_OURS,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.croma_s2_from_ours = list(croma_s2_from_ours)
        self.s1_indices = list(s1_indices)
        self.our_s2_indices_used = sorted(i for i in self.croma_s2_from_ours if i >= 0)
        self.max_required_band = max(self.our_s2_indices_used + self.s1_indices)
        self.allowed = set(allowed_asset_types)
        with (self.dataset_root / 'manifest.json').open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'
        records = []
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed: continue
            img_file = r.get('image_file')
            if not img_file: continue
            p = images_dir / img_file
            if not p.exists(): continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1: continue
            except Exception:
                continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if not records:
            raise RuntimeError(f'no records: {dataset_root}')
        self.records = records

    def __len__(self):
        return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        H, W = arr.shape[1], arr.shape[2]
        s2_ours = arr[self.our_s2_indices_used, :, :]
        s2_ours = percentile_normalize(s2_ours)
        ours_pos = {our_idx: pos for pos, our_idx in enumerate(self.our_s2_indices_used)}
        s2_croma = np.zeros((12, H, W), dtype=np.float32)
        for slot, our_idx in enumerate(self.croma_s2_from_ours):
            if our_idx >= 0:
                s2_croma[slot] = s2_ours[ours_pos[our_idx]]
        s1 = arr[self.s1_indices, :, :].astype(np.float32)
        s1 = percentile_normalize(s1)
        return np.concatenate([s2_croma, s1], axis=0)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)
        return {'image': torch.from_numpy(img),
                'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']
        img = F.interpolate(img.unsqueeze(0),
                            size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


# Build per-cell datasets.
source_datasets = {}
for region, sector, local in ready:
    base = CROMADataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


Built 28 cell datasets


In [8]:
# Load the spatial split artifact and slice each cell into train/val/test
# SubsetViews accordingly.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles in source_datasets have no split assignment '
          '(likely AlphaEarth-missing; will be excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')
print(f'  total in splits: {len(train_global) + len(val_global) + len(test_global)}')


Loaded split artifact: 18,750 asset_id -> split entries
  splits distribution: Counter({'train': 13087, 'val': 2851, 'test': 2812})

Global: train=13087  val=2856  test=2813
  total in splits: 18756


In [9]:
# Diagnostic: regenerate the old v1 random-stratified split inside the
# notebook for an exact-match comparison. Same logic as v1 CROMA's
# stratified_split (per-cell, by-class, seed=42).
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)

def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


# Walk cells in the same sorted (region, sector) order as v1 builds source_datasets.
old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

# Compare against asset_to_split (the new spatial split).
common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')


Diagnostic: train/val/test transition table (old random -> new spatial)
Comparing on 18,750 tiles in both old and new splits

old \ new      train       val       test
--------------------------------------------------
train           9120      1983       1978
val             1946       410        415
test            2021       458        419

Unchanged: 9,949 (53.1%)
Changed:   8,801 (46.9%)


In [10]:
import torch.nn as nn

class CROMABackbone(nn.Module):
    NAME = 'croma_base'
    def __init__(self, weights_path, image_resolution=IMAGE_SIZE, freeze=True):
        super().__init__()
        self.model = PretrainedCROMA(
            pretrained_path=weights_path, size='base',
            modality='both', image_resolution=image_resolution,
        )
        if freeze:
            self.model.eval()
            for p in self.model.parameters():
                p.requires_grad = False
        self.feature_dim = self._infer_feature_dim()

    def _split(self, x):
        return x[:, :12], x[:, 12:14]

    def _infer_feature_dim(self):
        device = next(self.model.parameters()).device
        dummy = torch.zeros(1, 14, IMAGE_SIZE, IMAGE_SIZE, device=device)
        s2, s1 = self._split(dummy)
        with torch.no_grad():
            out = self.model(SAR_images=s1, optical_images=s2)
        if not isinstance(out, dict) or 'joint_GAP' not in out:
            raise RuntimeError(f'Unexpected CROMA output keys: {list(out.keys()) if isinstance(out, dict) else type(out)}')
        d = out['joint_GAP'].shape[-1]
        print(f'  feature_dim (joint_GAP) = {d}')
        return d

    def forward(self, x):
        s2, s1 = self._split(x)
        out = self.model(SAR_images=s1, optical_images=s2)
        return out['joint_GAP']


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


print('CROMABackbone + InfraBenchClassifier defined.')


CROMABackbone + InfraBenchClassifier defined.


In [11]:
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """v2 per-sector F1 — macro-avg of per-class F1s over the sector's
    classes, computed on the FULL test set. Replaces the v1 in-sector-
    filtered grouped_f1 (which was buggy — see per_sector_f1_catchall.ipynb)."""
    out = {}
    cm = np.array(cm)
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        # per_region unchanged (each tile has a unique region label).
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition). v1 used '
            'a grouped_f1 that filtered samples to in-sector before computing '
            'macro_f1 — that version is deprecated and not reported here.'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set):
    set_seed(seed)
    print(f'\n--- seed {seed} ---')
    backbone = CROMABackbone(weights_path=CROMA_WEIGHTS,
                             image_resolution=IMAGE_SIZE, freeze=True)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=LP_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=LP_LR, weight_decay=1e-4)

    run_name = f'croma_v2_seed{seed}_linear_probe'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt = ckpt_dir / 'checkpoint_final.pt'

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(LP_EPOCHS):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })
        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': LP_EPOCHS,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST =================================
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)
    return {
        'run_name':     run_name,
        'backbone':     backbone.NAME,
        'condition':    'linear_probe',
        'num_epochs':   LP_EPOCHS,
        'seed':         seed,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).')


Device: cuda
Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).


In [12]:
# ============================================================================
# Default SMOKE_ONLY=True. Set False and re-run this cell + the next to train.
# ============================================================================
SMOKE_ONLY = False

print('Loading CROMA_base for smoke check...')
set_seed(SEEDS[0])
sb = CROMABackbone(weights_path=CROMA_WEIGHTS, image_resolution=IMAGE_SIZE, freeze=True)
sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
print(f'  feature_dim = {sb.feature_dim}')
print(f'  trainable params = {sum(p.numel() for p in sm.parameters() if p.requires_grad):,}')

smoke_loader = DataLoader(train_global, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate)
batch = next(iter(smoke_loader))
img = batch['image']
print(f'  Batch shape: {tuple(img.shape)}  (expect [4, 14, {IMAGE_SIZE}, {IMAGE_SIZE}])')
sm.eval()
with torch.no_grad():
    logits = sm(img.to(DEVICE))
print(f'  Logits shape: {tuple(logits.shape)}, range [{logits.min().item():.4f}, '
      f'{logits.max().item():.4f}], finite={torch.isfinite(logits).all().item()}')

print(f'\nSmoke OK. SMOKE_ONLY = {SMOKE_ONLY}  — multi-seed cell below will '
      f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


Loading CROMA_base for smoke check...
Initializing SAR encoder
Initializing optical encoder
Initializing joint SAR-optical encoder
  feature_dim (joint_GAP) = 768
  feature_dim = 768
  trainable params = 9,997
  Batch shape: (4, 14, 224, 224)  (expect [4, 14, 224, 224])
  Logits shape: (4, 13), range [-1.0335, 1.0371], finite=True

Smoke OK. SMOKE_ONLY = False  — multi-seed cell below will run all 3 seeds.


In [ ]:
# ============================================================================
# Multi-seed invocation. Trains 3 seeds sequentially, saves a JSON per seed,
# computes aggregate stats, writes confusion-matrix PNG. Gated by SMOKE_ONLY.
# ============================================================================
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping all training. Set False and re-run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'croma_v2_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'linear_probe': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    # ---- Load all 3 completed seeds from disk for aggregate ------------
    ALL_SEEDS = [314, 271, 161]
    per_seed_results = {}
    for seed in ALL_SEEDS:
        json_path = Path(OUTPUT_DIR) / f'croma_v2_seed{seed}_results.json'
        with open(json_path) as f:
            per_seed_results[seed] = _json.load(f)['linear_probe']
    SEEDS = ALL_SEEDS  # rebind so the rest of the aggregate references all 3

    # ---- Aggregate stats with per_seed arrays ---------------------------
    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {
            'mean':     float(arr.mean()),
            'std':      float(arr.std(ddof=0)),
            'per_seed': [float(v) for v in arr],
        }

    agg = {}
    # macro_f1 + acc
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    # per-class
    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {
            'class':    CLASS_NAMES[i],
            'idx':      i,
            'mean_f1':  float(per_class_arr[:, i].mean()),
            'std_f1':   float(per_class_arr[:, i].std(ddof=0)),
            'per_seed': [float(v) for v in per_class_arr[:, i]],
        }
        for i in range(len(CLASS_NAMES))
    ]

    # per-sector (v2)
    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s_per_seed = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1']
                        for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n':                n_seed,        # stable across seeds (same test set)
            'mean_macro_f1':    float(np.mean(f1s_per_seed)),
            'std_macro_f1':     float(np.std(f1s_per_seed, ddof=0)),
            'per_seed':         [float(v) for v in f1s_per_seed],
        }

    # per-region (in-region subset macro-F1)
    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s_per_seed = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
                        for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n':                n_seed,
            'mean_macro_f1':    float(np.nanmean(f1s_per_seed)),
            'std_macro_f1':     float(np.nanstd(f1s_per_seed, ddof=0)),
            'per_seed':         [float(v) for v in f1s_per_seed],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']               = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']   = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    agg_path = Path(OUTPUT_DIR) / 'croma_v2_aggregate.json'
    with open(agg_path, 'w') as f:
        _json.dump(agg, f, indent=2)
    print(f'\nAggregate saved: {agg_path}')

    # ---- Aggregate confusion matrix (sum across seeds) ------------------
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm),
                         where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'CROMA v2 spatial — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, '
                 f'seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / 'confusion_matrix_croma_v2_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    # ---- Summary print ---------------------------------------------------
    print('\n' + '=' * 76)
    print('CROMA v2 spatial — aggregate (3 seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- '
          f'{agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- '
          f'{agg["test_accuracy"]["std"]:.4f}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')


--- seed 161 ---
Initializing SAR encoder
Initializing optical encoder
Initializing joint SAR-optical encoder
  feature_dim (joint_GAP) = 768
  ep   1  loss=2.2834  val_acc=0.1338  val_f1=0.1041 *
  ep   2  loss=2.1660  val_acc=0.2311  val_f1=0.1414 *
  ep   3  loss=2.0983  val_acc=0.3211  val_f1=0.1704 *
  ep   4  loss=2.0631  val_acc=0.2766  val_f1=0.1827 *
  ep   5  loss=2.0437  val_acc=0.1891  val_f1=0.1543
  ep   6  loss=2.0291  val_acc=0.2062  val_f1=0.1722
  ep   7  loss=2.0152  val_acc=0.1677  val_f1=0.1326
  ep   8  loss=1.9999  val_acc=0.3113  val_f1=0.2126 *
  ep   9  loss=1.9944  val_acc=0.3120  val_f1=0.2145 *
  ep  10  loss=1.9914  val_acc=0.2539  val_f1=0.1700
  ep  11  loss=1.9895  val_acc=0.3382  val_f1=0.2058
  ep  12  loss=1.9616  val_acc=0.2381  val_f1=0.1766
  ep  13  loss=1.9648  val_acc=0.2405  val_f1=0.1983
  ep  14  loss=1.9494  val_acc=0.3505  val_f1=0.2277 *
  ep  15  loss=1.9658  val_acc=0.2791  val_f1=0.1934
  ep  16  loss=1.9643  val_acc=0.2714  val_f1=0.